# Slotfill on a Colab A100, open-weight models only

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/aous1998/slotfill/blob/main/notebooks/slotfill_colab_a100.ipynb)

**Runtime → Change runtime type → A100 GPU** (Colab Pro). No API keys needed: every system is an open-weight model running on this GPU.

| # | System | How it runs |
|---|--------|-------------|
| 1 | Qwen2.5-32B-Instruct (AWQ 4-bit), prompted | vLLM |
| 2 | Qwen2.5-32B-Instruct (AWQ 4-bit), structured output | vLLM guided JSON decoding |
| 3 | Qwen2.5-3B base, prompted | vLLM |
| 4 | Qwen2.5-3B base, structured output | vLLM guided JSON decoding |
| 5 | Qwen2.5-3B + LoRA r16 (424 docs) | vLLM + adapter |
| 6 | Qwen2.5-3B + LoRA r16 (100 docs) | vLLM + adapter |

The 32B model is the large baseline. The question is the same: can a small fine-tuned model match a much bigger prompted model?

Steps: 1) setup + tests, 2) train both adapters, 3) **restart the session**, 4) serve the small models and predict, 5) serve the 32B model and predict, 6) score everything.
Training and serving run in separate sessions because Unsloth and vLLM pin different library versions. Files in `/content` survive a session restart.

## 1. Setup

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import os
BRANCH = "colab-a100-open-weights"  # change to "main" once this branch is merged
if not os.path.exists("/content/slotfill"):
    !git clone -q -b {BRANCH} https://github.com/aous1998/slotfill.git /content/slotfill
%cd /content/slotfill
!pip install -q -r requirements.txt
!python -m pytest -q

In [ ]:
# The split is frozen (its sha256 is checked by the tests). The audit shows how much
# of each gold field is recoverable from the OCR text, i.e. the ceiling for any model.
!python scripts/audit_labels.py

## 2. Train

A few minutes per adapter on an A100. The hyperparameters are the same as the T4 notebook so results stay comparable.
Each run is logged in `adapters/<name>/train_log.json`.

In [ ]:
!pip install -q unsloth

In [ ]:
!python scripts/train.py --rank 16 --alpha 32 --lr 2e-4 --epochs 3 --out adapters/slotfill-r16

In [ ]:
# Ablation: same settings, only 100 training receipts.
!python scripts/train.py --rank 16 --alpha 32 --lr 2e-4 --epochs 3 --n-train 100 --out adapters/slotfill-n100

In [ ]:
import json, glob
for path in sorted(glob.glob("adapters/*/train_log.json")):
    log = json.load(open(path))
    evals = [h["eval_loss"] for h in log["log_history"] if "eval_loss" in h]
    c = log["config"]
    print(f"{path.split('/')[1]:16s} r={c['rank']} docs={log['train_docs']} gpu={log['gpu']} "
          f"time={log['train_seconds']/60:.1f}min peak={log['peak_gpu_mem_gb']}GB dev_loss/epoch={[round(e, 3) for e in evals]}")

## 3. Serve with vLLM

**Runtime → Restart session now**, then continue from here. You don't need to re-run the cells above.

In [ ]:
%cd /content/slotfill
!pip install -q vllm -r requirements.txt

In [ ]:
# Helpers: start a vLLM server in the background, wait until it answers, stop it and free the GPU.
import os, signal, subprocess, time, urllib.request

def start_server(base, no_lora=False, gpu_util="0.85", timeout_s=1200):
    env = dict(os.environ, BASE=base, GPU_UTIL=gpu_util, NO_LORA="1" if no_lora else "0")
    proc = subprocess.Popen(["bash", "scripts/serve_vllm.sh"], env=env, start_new_session=True,
                            stdout=open("vllm.log", "w"), stderr=subprocess.STDOUT)
    t0 = time.time()
    while time.time() - t0 < timeout_s:
        if proc.poll() is not None:
            raise RuntimeError("vLLM exited:\n" + open("vllm.log").read()[-3000:])
        try:
            print(urllib.request.urlopen("http://localhost:8000/v1/models").read().decode()[:400])
            print(f"ready in {time.time() - t0:.0f}s")
            return proc
        except Exception:
            time.sleep(5)
    stop_server(proc)
    raise TimeoutError(open("vllm.log").read()[-3000:])

def stop_server(proc):
    try:
        os.killpg(proc.pid, signal.SIGTERM)
        proc.wait(timeout=60)
    except Exception:
        os.killpg(proc.pid, signal.SIGKILL)
    time.sleep(5)
    !nvidia-smi --query-gpu=memory.used --format=csv

# What an A100 costs per hour if you rented it (roughly $1.2-1.6/h on RunPod/Lambda/Vast).
# Cost per 1k docs = this rate x the wall-clock time of each run.
GPU_HOURLY = 1.30

### 3a. Small model: Qwen2.5-3B base + both adapters on one server

In [ ]:
SMALL = "Qwen/Qwen2.5-3B-Instruct"
server = start_server(SMALL)
!python scripts/predict.py --provider vllm --model {SMALL}                 --gpu-hourly {GPU_HOURLY} --out results/preds_base.jsonl
!python scripts/predict.py --provider vllm --model {SMALL} --structured    --gpu-hourly {GPU_HOURLY} --out results/preds_base_structured.jsonl
!python scripts/predict.py --provider vllm --model slotfill-r16            --gpu-hourly {GPU_HOURLY} --out results/preds_lora_r16.jsonl
!python scripts/predict.py --provider vllm --model slotfill-n100           --gpu-hourly {GPU_HOURLY} --out results/preds_lora_n100.jsonl
stop_server(server)

### 3b. Large open-weight baseline: Qwen2.5-32B-Instruct (AWQ)

About 19 GB of weights, so it fits a 40 GB A100 with room for the KV cache. The first run downloads it, which takes a few minutes.
To try a different model, change `LARGE`. For example, `Qwen/Qwen2.5-14B-Instruct` runs at full precision, and
`Qwen/Qwen2.5-72B-Instruct-AWQ` needs an 80 GB A100.

In [ ]:
LARGE = "Qwen/Qwen2.5-32B-Instruct-AWQ"
server = start_server(LARGE, no_lora=True, gpu_util="0.90")
!python scripts/predict.py --provider vllm --model {LARGE}              --gpu-hourly {GPU_HOURLY} --out results/preds_large.jsonl
!python scripts/predict.py --provider vllm --model {LARGE} --structured --gpu-hourly {GPU_HOURLY} --out results/preds_large_structured.jsonl
stop_server(server)

## 4. Results

In [ ]:
!python scripts/evaluate.py \
  --run "Qwen2.5-32B AWQ, prompted=results/preds_large.jsonl" \
  --run "Qwen2.5-32B AWQ, structured output=results/preds_large_structured.jsonl" \
  --run "Qwen2.5-3B base, prompted=results/preds_base.jsonl" \
  --run "Qwen2.5-3B base, structured output=results/preds_base_structured.jsonl" \
  --run "Qwen2.5-3B + LoRA r16 (424 docs)=results/preds_lora_r16.jsonl" \
  --run "Qwen2.5-3B + LoRA r16 (100 docs)=results/preds_lora_n100.jsonl"

In [ ]:
# Look at what each system gets wrong before drawing conclusions.
import json, glob
for path in sorted(glob.glob("results/errors_*.jsonl")):
    errs = [json.loads(l) for l in open(path)]
    print(f"\n=== {path}  ({len(errs)} docs with at least one wrong field)")
    for e in errs[:3]:
        print(e["id"], e["error"], e["wrong"], "\n  gold:", e["gold"], "\n  got: ", e["raw_output"])

In [ ]:
# Keep the results: download them, then commit results/ to the repo.
!zip -qr slotfill_results.zip results adapters/*/train_log.json adapters/*/adapter_config.json
from google.colab import files
files.download("slotfill_results.zip")